# Scraper USP Primo — **Revisado (balizado no HTML anexado)**

Notebook focado


In [1]:
import os, re, csv, time
import requests
import pandas as pd
from typing import Optional, List, Dict, Tuple, Set
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urlparse, urljoin, parse_qs, urlencode, urlunparse

BASE = "https://buscaintegrada.usp.br"
BASE_NON_URL = 'buscaintegrada'

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive",
})

In [2]:
# === Configuração (COLE SUA URL FILTRADA AQUI) ===
URL_FILTRADA = "https://buscaintegrada.usp.br/primo_library/libweb/action/search.do?fn=search&ct=search&initialSearch=true&mode=Advanced&tab=default_tab&indx=1&dum=true&srt=rank&vid=USP&tb=t&vl(4708870UI0)=sub&vl(1UIStartWith0)=sub&vl(1UIStartWith1)=contains&vl(freeText1)=sociologia%20das%20elites&vl(4708301UI6)=01&vl(4708302UI6)=01&vl(4708303UI6)=1970&vl(4708304UI6)=31&vl(4708305UI6)=12&vl(4708306UI6)=9999&Submit=Buscar"
THROTTLE = 0.6

In [3]:
def pausa(segundos: float = 0.6):
    if segundos > 0:
        time.sleep(segundos)

def http_get(url: str, timeout: int = 25, tentativas: int = 3) -> Optional[requests.Response]:
    for i in range(tentativas):
        try:
            r = SESSION.get(url, timeout=timeout, allow_redirects=True)
            if r.status_code == 200:
                return r
        except requests.RequestException:
            pass
        time.sleep(1.0 * (i + 1))
    return None

In [4]:

# === Parsing da LISTA (apenas esta página, com múltiplos seletores) ===
def extrair_itens_lista(html_texto: str) -> List[Dict]:
    soup = BeautifulSoup(html_texto, "html.parser")
    itens: List[Dict] = []
    vistos = set()

    seletores = [
        # Primo clássico
        "a.EXLResultTitle",
        "a.EXLBriefResultsDisplayTitle",
        "a[title='Detalhes']",
        "h2.EXLResultTitle a",
        # Primo novo (Angular) / variantes
        "h3 a[href*='display.do']",
        "a[href*='fulldisplay']",
        "a[class*='recordTitle']",
        "a[data-recordid]",
        "a[aria-label*='Detalhes']",
        "a[aria-label*='Details']",
        # Outros
        "div.EXLResultTitle a",
    ]

    links = []
    for sel in seletores:
        links.extend(soup.select(sel))

    # Fallback: qualquer <a> com padrão de registro e texto útil
    if not links:
        for a in soup.select("a[href]"):
            href = a.get("href") or ""
            txt = a.get_text(strip=True)
            if not txt:
                continue
            if any(k in href for k in ("display.do", "fulldisplay")) and len(txt) > 3:
                links.append(a)

    def normaliza(a_tag) -> Optional[Dict]:
        href = a_tag.get("href") or ""
        titulo = a_tag.get_text(strip=True)
        if not href or not titulo:
            return None
        url_registro = urljoin(BASE, href)
        if url_registro in vistos:
            return None
        vistos.add(url_registro)
        return {"title": titulo, "record_url": url_registro}

    for a in links:
        info = normaliza(a)
        if not info:
            continue
        low = info["title"].lower()
        if any(x in low for x in ["próximo", "anterior", "detalhes", "refinar", "acesso", "login", "entrar"]):
            continue
        itens.append(info)

    return itens


In [5]:
# Optional: use dateutil if available (handles many formats)
try:
    from dateutil import parser as dateparser
except Exception:
    dateparser = None

KEYS_LISTLIKE = {
    "author", "citation_author", "dc.creator",
    "keywords", "citation_keywords", "dc.subject"
}
KEYS_DATE = {
    "date", "citation_date", "citation_publication_date",
    "dc.date", "dc.date.issued", "dc.date.created", "prism.publicationdate"
}

def _norm_key(k: str) -> str:
    """Normalize meta key to a canonical snake_case-ish form."""
    k = (k or "").strip()
    if not k: return ""
    k = k.lower()
    # collapse common namespaces/aliases
    k = k.replace("http-equiv:", "").replace("property:", "").replace("name:", "").strip()
    # unify separators
    k = k.replace(":", ".")
    k = re.sub(r"[^a-z0-9.]+", "_", k)
    k = re.sub(r"_+", "_", k).strip("_")
    return k

def _split_keywords(val: str) -> list:
    parts = re.split(r"[;,/•|]\s*|\s{2,}", val or "")
    parts = [re.sub(r"^(palavras?-chave|keywords)\s*:\s*", "", p, flags=re.I).strip(" .;,-") 
             for p in parts if p and p.strip()]
    # dedupe case-insensitively
    seen, out = set(), []
    for p in parts:
        pl = p.lower()
        if pl not in seen:
            out.append(p); seen.add(pl)
    return out

def _parse_date(val: str) -> dict:
    """Return {'date_raw':..., 'date_iso':..., 'year':...} best-effort."""
    raw = (val or "").strip()
    if not raw:
        return {"date_raw": None, "date_iso": None, "year": None}
    # prefer dateutil if available
    if dateparser:
        try:
            dt = dateparser.parse(raw, fuzzy=True)
            return {
                "date_raw": raw,
                "date_iso": dt.date().isoformat(),
                "year": str(dt.year)
            }
        except Exception:
            pass
    # fallback: simple regexes
    m = re.search(r"\b(19|20)\d{2}-\d{2}-\d{2}\b", raw)
    if m:
        try:
            d = datetime.strptime(m.group(0), "%Y-%m-%d").date()
            return {"date_raw": raw, "date_iso": d.isoformat(), "year": str(d.year)}
        except Exception:
            pass
    m = re.search(r"\b(19|20)\d{2}\b", raw)
    return {"date_raw": raw, "date_iso": None, "year": m.group(0) if m else None}

def extract_and_normalize_meta(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    meta_out = {}

    # gather all <meta> tags (name / property / http-equiv)
    tags = soup.find_all("meta")
    for t in tags:
        key = t.get("name") or t.get("property") or t.get("http-equiv")
        val = t.get("content") or t.get("value")
        if not key or not val:
            continue
        k = _norm_key(key)
        if not k:
            continue

        # List-like keys (collect multiple values)
        is_listlike = (k in KEYS_LISTLIKE)
        is_keywords = (k in {"keywords", "citation_keywords", "dc.subject"})
        is_date = (k in KEYS_DATE)

        if is_keywords:
            vals = _split_keywords(val)
            if not vals:
                continue
            meta_out.setdefault("keywords", [])
            # extend while deduping
            for v in vals:
                if v not in meta_out["keywords"]:
                    meta_out["keywords"].append(v)
            continue

        if is_date:
            parsed = _parse_date(val)
            # keep the best we’ve seen so far
            meta_out.setdefault("date_raw", parsed["date_raw"])
            meta_out.setdefault("date_iso", parsed["date_iso"])
            meta_out.setdefault("year", parsed["year"])
            # prefer to upgrade if we later get a full date where we had only a year
            if parsed["date_iso"] and not meta_out.get("date_iso"):
                meta_out["date_iso"] = parsed["date_iso"]
            if parsed["year"] and not meta_out.get("year"):
                meta_out["year"] = parsed["year"]
            continue

        if is_listlike:
            meta_out.setdefault(k, [])
            v = val.strip()
            if v and v not in meta_out[k]:
                meta_out[k].append(v)
        else:
            # keep first seen if empty; otherwise don't overwrite
            meta_out.setdefault(k, val.strip())

    # Normalize some aliases into friendlier fields
    # authors
    if "author" in meta_out and isinstance(meta_out["author"], list):
        meta_out["authors"] = meta_out["author"]
    elif "citation_author" in meta_out:
        meta_out["authors"] = meta_out["citation_author"]

    # prefer keywords list as a single string if you want CSV-ready
    if "keywords" in meta_out and isinstance(meta_out["keywords"], list):
        meta_out["keywords_str"] = "; ".join(meta_out["keywords"])

    # common conveniences
    if "citation_pdf_url" in meta_out:
        meta_out["pdf_url"] = meta_out["citation_pdf_url"]
    if "citation_abstract" in meta_out and "abstract" not in meta_out:
        meta_out["abstract"] = meta_out["citation_abstract"]
    if "citation_title" in meta_out and "title" not in meta_out:
        meta_out["title"] = meta_out["citation_title"]

    return meta_out

In [6]:
def row_from_meta(meta: dict, fallback: dict | None = None) -> dict:
    """Flatten lists, apply fallbacks (e.g., item title/URL), and return a row dict."""
    fb = fallback or {}
    # authors / keywords may be lists; make them CSV-friendly
    authors  = meta.get("authors")
    if isinstance(authors, list):
        authors = "; ".join(a.strip() for a in authors if a and str(a).strip())
    keywords = meta.get("keywords")
    if isinstance(keywords, list):
        keywords = "; ".join(k.strip() for k in keywords if k and str(k).strip())

    return {
        "title":       meta.get("title") or fb.get("title"),
        "record_url":  fb.get("record_url") or meta.get("record_url"),
        "authors":     authors or meta.get("citation_author"),
        "year":        meta.get("year"),
        "date_iso":    meta.get("date_iso"),
        "keywords":    keywords or meta.get("keywords_str"),
        "abstract":    meta.get("abstract") or meta.get("citation_abstract"),
        "pdf_url":     meta.get("pdf_url") or meta.get("citation_pdf_url"),
        "full_data":   meta
    }

In [7]:
def extrair_url_proxima_pagina(html_text: str, current_url: str) -> Optional[str]:
    soup = BeautifulSoup(html_text, "html.parser")

    # 1) rel="next"
    a = soup.select_one("a[rel='next'], link[rel='next']")
    if a and a.get("href"):
        return urljoin(current_url, a["href"])

    # 2) Texto do link
    for tag in soup.select("a[href], button[href]"):
        txt = (tag.get_text(" ", strip=True) or "").lower()
        if txt in {"próximo", "proximo", "próx.", "next", "»", "›", ">"}:
            href = tag.get("href")
            if href:
                return urljoin(current_url, href)

    # 3) Primo clássico
    cand = soup.select_one("a.EXLNext, a[title*='Próximo'], a[title*='Next']")
    if cand and cand.get("href"):
        return urljoin(current_url, cand["href"])

    return None

def _atualiza_indx(url: str, novo_indx: int) -> str:
    pu = urlparse(url)
    qs = parse_qs(pu.query, keep_blank_values=True)
    qs["indx"] = [str(max(1, int(novo_indx)))]
    new_qs = urlencode(qs, doseq=True)
    return urlunparse((pu.scheme, pu.netloc, pu.path, pu.params, new_qs, pu.fragment))

def coletar_todas_paginas(
    url_inicial: str,
    throttle: float = 0.6,
    max_pages: Optional[int] = None,
    domain_whitelist: Optional[set] = None,
    skip_substrings: tuple = ("metalibsfx.aguia.usp.br",)
) -> List[Dict]:
    """
    Percorre TODAS as páginas de resultados e agrega itens
    **SEM deduplicação de registros** (duplicatas serão mantidas).
    - Aceita extrair_itens_lista() que retorna list OU (list, has_next)
    - Filtra itens com domínios indesejados (ex.: resolvers SFX) se desejar
    - Fallback de paginação aumentando 'indx' pelo tamanho da página (ou +10)
    """
    vistos_pag = set()           # mantém só para não reprocessar a MESMA página
    itens_totais: List[Dict] = []

    url = url_inicial
    pagina = 0
    while url:
        if url in vistos_pag:
            break
        vistos_pag.add(url)

        resp = http_get(url)
        if not resp:
            break

        # Compatível com ambas assinaturas de extrair_itens_lista
        ret = extrair_itens_lista(resp.text)
        if isinstance(ret, tuple):
            itens_page, _has_next = ret
        else:
            itens_page = ret

        # Filtros opcionais (NÃO removem duplicatas; apenas itens "lixo")
        filtrados = []
        for it in itens_page:
            href = it.get("record_url", "")
            if any(s in href for s in skip_substrings):
                continue
            if domain_whitelist:
                host = urlparse(href).netloc.lower()
                if not any(host.endswith(d) for d in domain_whitelist):
                    # mantenha se quiser ver tudo, comente este continue se não quiser filtrar
                    pass
            filtrados.append(it)

        itens_totais.extend(itens_page)

        pagina += 1
        print(f"[pág {pagina}] itens nesta página: {len(itens_page)} | adicionados: {len(filtrados)} | total: {len(itens_totais)}")

        if max_pages and pagina >= max_pages:
            break

        # 1) Tente link "Próximo"
        proxima = extrair_url_proxima_pagina(resp.text, url)

        # 2) Fallback: incremente 'indx'
        if not proxima:
            try:
                curr_indx = int(parse_qs(urlparse(url).query).get("indx", ["1"])[0])
            except Exception:
                curr_indx = 1
            passo = len(itens_page) if len(itens_page) > 0 else 10
            proxima = _atualiza_indx(url, curr_indx + passo)

            # Evita laço se não há itens
            if len(itens_page) == 0 and proxima == url:
                proxima = None

        url = proxima
        pausa(throttle)

    return itens_totais

In [8]:
itens = coletar_todas_paginas(
    URL_FILTRADA,
    throttle=THROTTLE,
    max_pages=None, 
    # Opcional: whiteliste alguns domínios finais desejados:
    # domain_whitelist={"revistas.usp.br", "teses.usp.br", "scielo.br", "repositorio.usp.br"}
)
print(f"TOTAL de itens coletados: {len(itens)}")
itens[:3]

[pág 1] itens nesta página: 9 | adicionados: 2 | total: 9
TOTAL de itens coletados: 9


[{'title': 'As estratégias de recomposição e rearticulaçãodaselitesdo grupo étnico alemão no Brasil pós-Estado Novo: Uma análise à luz dasociologiahistórica e dasociologiadaselites',
  'record_url': 'https://metalibsfx.aguia.usp.br:3443/sfxlcl41?ctx_ver=Z39.88-2004&ctx_enc=info:ofi/enc:UTF-8&ctx_tim=2025-10-12T15%3A22%3A41IST&url_ver=Z39.88-2004&url_ctx_fmt=infofi/fmt:kev:mtx:ctx&rfr_id=info:sid/primo.exlibrisgroup.com:primo3-Article-scielo_cross&rft_val_fmt=info:ofi/fmt:kev:mtx:journal&rft.genre=article&rft.atitle=As%20estrat%C3%A9gias%20de%20recomposi%C3%A7%C3%A3o%20e%20rearticula%C3%A7%C3%A3o%20das%20elites%20do%20grupo%20%C3%A9tnico%20alem%C3%A3o%20no%20Brasil%20p%C3%B3s-Estado%20Novo:%20Uma%20an%C3%A1lise%20%C3%A0%20luz%20da%20sociologia%20hist%C3%B3rica%20e%20da%20sociologia%20das%20elites&rft.jtitle=Civitas%20(Porto%20Alegre,%20Brazil)&rft.au=Voigt,%20Lucas&rft.date=2021-05-04&rft.volume=21&rft.issue=1&rft.spage=144&rft.epage=158&rft.pages=144-158&rft.issn=1519-6089&rft.eissn=19

In [17]:
rows = []
for item in itens: 
    rrec = http_get(item["record_url"])
    if not rrec:
        continue
    meta = extract_and_normalize_meta(rrec.text)  # your function from earlier
    rows.append(row_from_meta(meta, fallback=item))

df = pd.DataFrame(rows)
df

,title,record_url,authors,year,date_iso,keywords,abstract,pdf_url,full_data
0,As estratégias de recomposição e rearticulação...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."
1,Espaço social e redes: contribuições metodológ...,https://www.revistas.usp.br/ts/article/view/12...,Elisa Klüger,2017,2017-12-12,Elites; Redes sociais; Análise de correspondên...,O artigo apresenta uma contribuição metodológi...,https://revistas.usp.br/ts/article/download/12...,"{'viewport': 'width=device-width, initial-scal..."
2,As estratégias de recomposição e rearticulação...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."
3,"Sociologia das reformas: orquestrando crítica,...",https://www.revistas.usp.br/ts/article/view/23...,Ana Paula Hey; Francesco Tomei; Pedro Louro,2024,2024-12-10,Sociology of elites; Reform; Expertise; Public...,Introdução ao dossiê Sociologia da reforma,https://revistas.usp.br/ts/article/download/23...,"{'viewport': 'width=device-width, initial-scal..."
4,Espaço social e redes: contribuições metodológ...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."
...,...,...,...,...,...,...,...,...,...
73,O estudo de caso nasociologiados tribunais: o ...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."
74,Prosopografía: estado del conocimiento sobre e...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."
75,The Force of Law: research routes in sociology...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."
76,Sociologiae história na obra de José de Souza ...,https://metalibsfx.aguia.usp.br:3443/sfxlcl41?...,None,None,None,None,None,None,"{'content_type': 'text/html; charset=UTF-8', '..."


In [10]:
df.to_csv(f'{os.getcwd()}/out_simples/{BASE_NON_URL}_primeira_coleta_{str(datetime.today().date())}.csv')

In [14]:
itens = coletar_todas_paginas(
    URL_FILTRADA,
    throttle=THROTTLE,
    force_bulk_size=50,    # tenta 50 por página
)

[pág 1] itens nesta página: 9 | adicionados: 2 | total: 2
